# Regression pour maintenance predictive

Objectif : rendre la regression plus interessante en ajoutant des flags metier de degradation, puis predire un RUL (`Remaining Useful Life`) plus coherent.

Le notebook ne modifie pas les CSV propres existants. Il genere un dataset enrichi dans `artifacts/regression_dataset_with_flags.csv`.

In [2]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

DATA_PATH = "data/clean/scada_capteurs_propre.csv"
BUSINESS_DATA_PATH = "data/processed/business/scada_capteurs_business.csv"
RAW_DATA_PATH = "data/raw/scada_capteurs_sale.csv"
ARTIFACT_DIR = "artifacts"

os.makedirs(ARTIFACT_DIR, exist_ok=True)
pd.set_option("display.max_columns", 80)

## 1. Chargement des donnees

On repart du fichier SCADA propre. Si les timestamps propres sont mal reconvertis en 1970, on reprend les timestamps bruts pour calculer les dates de maintenance.

In [3]:
df = pd.read_csv(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")

if df["timestamp"].dt.year.median() < 2000 and os.path.exists(RAW_DATA_PATH):
    raw_timestamps = pd.read_csv(RAW_DATA_PATH, usecols=["timestamp"])
    df["timestamp"] = pd.to_datetime(raw_timestamps["timestamp"], errors="coerce").fillna(df["timestamp"])

print(df.shape)
df.head()

(100000, 33)


,timestamp,machine_id,cycle_time_sec,Temperature_C,Vibration_mms,Sound_dB,Oil_Level_pct,Coolant_Level_pct,Hydraulic_Pressure_bar,Coolant_Flow_L_min,Heat_Index,Power_Consumption_kW,Operational_Hours,Error_Codes_Last_30_Days,sensor_anomaly_score,AI_Override_Events,flag_temperature_high,flag_vibration_high,flag_sound_high,flag_oil_low,flag_coolant_low,flag_pressure_low,flag_coolant_flow_low,flag_heat_high,flag_power_high,flag_error_codes_high,flag_anomaly_high,degradation_score,degradation_rate_pct,predicted_failure_probability,Remaining_Useful_Life_days,Failure_Within_7_Days,Maintenance_Required_Within_45_Days
0,2026-01-01 06:00:00,M01,45.07,60.74,1.43,67.64,93.72,85.07,117.79,33.50,61.39,37.44,921.970,1.0,0.000,2.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.313,328.1,0,0.0
1,2026-01-01 06:00:00,M02,45.60,61.85,2.32,75.63,85.69,67.32,136.44,30.88,65.09,44.20,1030.205,7.0,0.227,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,3.2,21.1,0.313,215.3,0,0.0
2,2026-01-01 06:00:00,M03,44.71,53.30,1.76,73.92,76.87,93.78,121.04,36.07,72.22,41.43,1138.440,2.0,0.157,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.029,285.6,0,0.0
3,2026-01-01 06:00:00,M04,50.95,75.04,3.51,76.95,73.95,69.88,113.29,24.86,73.01,51.38,1594.710,1.0,0.567,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.6,23.7,0.359,179.8,0,0.0
4,2026-01-01 06:05:00,M01,47.29,60.94,2.13,68.79,87.17,78.73,135.94,33.50,68.75,41.92,925.030,2.0,0.257,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.322,301.6,0,0.0


## 2. Creation des flags de degradation

Chaque flag represente un signal industriel anormal : vibration elevee, temperature elevee, huile basse, liquide de refroidissement bas, pression basse, etc.

Ces flags rendent le probleme plus lisible : le modele apprend a relier des symptomes de degradation a un RUL plus faible.

In [4]:
df_flags = df.copy()

df_flags["flag_temperature_high"] = (df_flags["Temperature_C"] > 68).astype(int)
df_flags["flag_vibration_high"] = (df_flags["Vibration_mms"] > 2.6).astype(int)
df_flags["flag_sound_high"] = (df_flags["Sound_dB"] > 82).astype(int)
df_flags["flag_oil_low"] = (df_flags["Oil_Level_pct"] < 78).astype(int)
df_flags["flag_coolant_low"] = (df_flags["Coolant_Level_pct"] < 72).astype(int)
df_flags["flag_pressure_low"] = (df_flags["Hydraulic_Pressure_bar"] < 115).astype(int)
df_flags["flag_coolant_flow_low"] = (df_flags["Coolant_Flow_L_min"] < 25).astype(int)
df_flags["flag_heat_high"] = (df_flags["Heat_Index"] > 75).astype(int)
df_flags["flag_power_high"] = (df_flags["Power_Consumption_kW"] > 55).astype(int)
df_flags["flag_error_codes_high"] = (df_flags["Error_Codes_Last_30_Days"] >= 3).astype(int)
df_flags["flag_anomaly_high"] = (df_flags["sensor_anomaly_score"] >= 0.70).astype(int)
df_flags["flag_operational_hours_high"] = (df_flags["Operational_Hours"] >= 1200).astype(int)

flag_columns = [col for col in df_flags.columns if col.startswith("flag_")]
df_flags[flag_columns].mean().sort_values(ascending=False)

flag_operational_hours_high    0.89416
flag_vibration_high            0.62258
flag_error_codes_high          0.60881
flag_oil_low                   0.57369
flag_temperature_high          0.56764
flag_coolant_low               0.38213
flag_heat_high                 0.35111
flag_pressure_low              0.28705
flag_power_high                0.15366
flag_coolant_flow_low          0.14463
flag_anomaly_high              0.08116
flag_sound_high                0.06171
dtype: float64

## 3. Construction d'un taux de degradation metier

On pondere les flags selon leur importance. Par exemple, les vibrations, la temperature, l'huile basse et les erreurs recentes pesent plus lourd.

Le taux obtenu est compris entre 0 et 100%.

In [5]:
weights = {
    "flag_vibration_high": 2.0,
    "flag_temperature_high": 1.6,
    "flag_oil_low": 1.5,
    "flag_coolant_low": 1.4,
    "flag_pressure_low": 1.4,
    "flag_error_codes_high": 1.8,
    "flag_anomaly_high": 1.5,
    "flag_heat_high": 1.2,
    "flag_power_high": 1.0,
    "flag_sound_high": 0.8,
    "flag_coolant_flow_low": 1.0,
    "flag_operational_hours_high": 1.0,
}

weighted_score = sum(df_flags[col] * weight for col, weight in weights.items())
max_score = sum(weights.values())

df_flags["degradation_score"] = weighted_score.round(2)
df_flags["degradation_rate_pct"] = (weighted_score / max_score * 100).clip(0, 100).round(1)

df_flags[["machine_id", "degradation_score", "degradation_rate_pct"] + flag_columns].head()

,machine_id,degradation_score,degradation_rate_pct,flag_temperature_high,flag_vibration_high,flag_sound_high,flag_oil_low,flag_coolant_low,flag_pressure_low,flag_coolant_flow_low,flag_heat_high,flag_power_high,flag_error_codes_high,flag_anomaly_high,flag_operational_hours_high
0,M01,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0
1,M02,3.2,19.8,0,0,0,0,1,0,0,0,0,1,0,0
2,M03,1.5,9.3,0,0,0,1,0,0,0,0,0,0,0,0
3,M04,9.9,61.1,1,1,0,1,1,1,1,0,0,0,0,1
4,M01,0.0,0.0,0,0,0,0,0,0,0,0,0,0,0,0


## 4. Creation d'une cible RUL plus coherente

Dans le CSV initial, `Remaining_Useful_Life_days` est peu correle aux capteurs. Pour tester la regression, on cree donc une cible experimentale :

- plus la degradation est forte, plus le RUL baisse ;
- si une panne est indiquee dans les 7 jours, on force un RUL plus court ;
- on ajoute un bruit leger pour garder un comportement realiste.

In [6]:
rng = np.random.default_rng(42)

base_rul = 365 - (df_flags["degradation_rate_pct"] / 100 * 330)
wear_penalty = ((df_flags["Operational_Hours"] - df_flags["Operational_Hours"].min()) /
                (df_flags["Operational_Hours"].max() - df_flags["Operational_Hours"].min()) * 35)
noise = rng.normal(0, 8, len(df_flags))

df_flags["engineered_rul_days"] = (base_rul - wear_penalty + noise).clip(1, 365)

if "Failure_Within_7_Days" in df_flags.columns:
    failure_mask = df_flags["Failure_Within_7_Days"].astype(int) == 1
    df_flags.loc[failure_mask, "engineered_rul_days"] = df_flags.loc[failure_mask, "engineered_rul_days"].clip(1, 45)

df_flags["engineered_rul_days"] = df_flags["engineered_rul_days"].round(1)
df_flags["engineered_maintenance_date"] = df_flags["timestamp"] + pd.to_timedelta(
    np.ceil(df_flags["engineered_rul_days"]), unit="D"
)

df_flags[["machine_id", "timestamp", "degradation_rate_pct", "engineered_rul_days", "engineered_maintenance_date"]].head(10)

,machine_id,timestamp,degradation_rate_pct,engineered_rul_days,engineered_maintenance_date
0,M01,2026-01-01 06:00:00,0.0,365.0,2027-01-01 06:00:00
1,M02,2026-01-01 06:00:00,19.8,288.6,2026-10-17 06:00:00
2,M03,2026-01-01 06:00:00,9.3,335.0,2026-12-02 06:00:00
3,M04,2026-01-01 06:00:00,61.1,154.6,2026-06-05 06:00:00
4,M01,2026-01-01 06:05:00,0.0,349.2,2026-12-17 06:05:00
5,M02,2026-01-01 06:05:00,18.5,284.1,2026-10-13 06:05:00
6,M03,2026-01-01 06:05:00,27.8,268.5,2026-09-27 06:05:00
7,M04,2026-01-01 06:05:00,32.7,238.7,2026-08-28 06:05:00
8,M01,2026-01-01 06:10:00,0.0,364.3,2027-01-01 06:10:00
9,M02,2026-01-01 06:10:00,6.2,328.3,2026-11-26 06:10:00


## 5. Entrainement des modeles de regression

La cible predite devient `engineered_rul_days`.

On ajoute les flags aux features numeriques, car ils rendent les signaux de degradation explicites.

In [7]:
numeric_features = [
    "cycle_time_sec",
    "Temperature_C",
    "Vibration_mms",
    "Sound_dB",
    "Oil_Level_pct",
    "Coolant_Level_pct",
    "Hydraulic_Pressure_bar",
    "Coolant_Flow_L_min",
    "Heat_Index",
    "Power_Consumption_kW",
    "Operational_Hours",
    "Error_Codes_Last_30_Days",
    "sensor_anomaly_score",
]

features = numeric_features + flag_columns + ["degradation_score", "degradation_rate_pct"]
target = "engineered_rul_days"

X = df_flags[features]
y = df_flags[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

models = {
    "LinearRegression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]),
    "RandomForestRegressor": RandomForestRegressor(
        n_estimators=300,
        random_state=42,
        min_samples_leaf=3,
    ),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42),
}

results = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        "model": name,
        "mae_days": mean_absolute_error(y_test, pred),
        "rmse_days": np.sqrt(mean_squared_error(y_test, pred)),
        "r2_score": r2_score(y_test, pred),
    })
    fitted_models[name] = model

results_df = pd.DataFrame(results).sort_values("mae_days")
results_df

,model,mae_days,rmse_days,r2_score
0,LinearRegression,6.922374,11.166817,0.984107
2,GradientBoostingRegressor,6.980395,11.153323,0.984146
1,RandomForestRegressor,7.206803,11.416051,0.983390


## 5 bis. Comparaison clean vs donnees business

On compare le modele technique actuel avec une variante enrichie par les colonnes metier issues de `data/processed/business/scada_capteurs_business.csv`.

Objectif : verifier si le contexte industriel (`machine_type`, `production_line_id`, `criticality_level`, maturite du site, qualite des donnees, fiabilite capteurs, etc.) ameliore vraiment la regression, sans remplacer les signaux capteurs.


In [8]:
if not os.path.exists(BUSINESS_DATA_PATH):
    raise FileNotFoundError(
        f"Fichier introuvable : {BUSINESS_DATA_PATH}. "
        "Lance avant : python scripts/enrich_mecha_business_data.py"
    )

business_df = pd.read_csv(BUSINESS_DATA_PATH)

# Le fichier business est genere ligne a ligne depuis le SCADA propre :
# on reprend donc la cible et les flags deja construits dans df_flags.
business_model_df = df_flags.copy()
business_columns = [
    "old_machine_id",
    "machine_id",
    "machine_name",
    "machine_type",
    "factory_id",
    "factory_name",
    "country",
    "production_line_id",
    "production_line_name",
    "product_family",
    "manufacturer",
    "criticality_level",
]
available_business_columns = [col for col in business_columns if col in business_df.columns]

for col in available_business_columns:
    business_model_df[col] = business_df[col].astype("string").fillna("unknown")

# On evite les identifiants trop specifiques dans les features du modele.
business_categorical_features = [
    "machine_type",
    "factory_id",
    "production_line_id",
    "product_family",
    "manufacturer",
    "criticality_level",
    "machine_generation",
    "site_maturity_level",
    "equipment_level",
    "sensor_reliability",
    "data_quality_level",
    "maintenance_strategy",
    "machine_availability_profile",
    "oee_profile",
    "unplanned_downtime_profile",
    "quality_drift_profile",
    "energy_efficiency_profile",
    "systems_maturity",
]
business_categorical_features = [
    col for col in business_categorical_features if col in business_model_df.columns
]

X_clean = df_flags[features]
X_business = business_model_df[features + business_categorical_features]
y_business = business_model_df[target]

X_clean_train, X_clean_test, y_clean_train, y_clean_test = train_test_split(
    X_clean, y_business, test_size=0.2, random_state=42
)
X_business_train, X_business_test, y_business_train, y_business_test = train_test_split(
    X_business, y_business, test_size=0.2, random_state=42
)

preprocessor_business = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), features),
        ("business", OneHotEncoder(handle_unknown="ignore"), business_categorical_features),
    ]
)

comparison_models = {
    "clean_linear_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression()),
    ]),
    "business_linear_regression": Pipeline([
        ("preprocessor", preprocessor_business),
        ("model", LinearRegression()),
    ]),
    "clean_gradient_boosting": GradientBoostingRegressor(random_state=42),
}

business_results = []

for model_name, model in comparison_models.items():
    if model_name.startswith("business"):
        model.fit(X_business_train, y_business_train)
        pred = model.predict(X_business_test)
        dataset_type = "business"
    else:
        model.fit(X_clean_train, y_clean_train)
        pred = model.predict(X_clean_test)
        dataset_type = "clean"

    business_results.append({
        "dataset_type": dataset_type,
        "model": model_name,
        "mae_days": mean_absolute_error(y_business_test, pred),
        "rmse_days": np.sqrt(mean_squared_error(y_business_test, pred)),
        "r2_score": r2_score(y_business_test, pred),
        "business_features_used": ", ".join(business_categorical_features) if dataset_type == "business" else "",
    })

business_comparison_df = pd.DataFrame(business_results).sort_values("mae_days")
business_comparison_path = os.path.join(ARTIFACT_DIR, "regression_clean_vs_business_comparison.csv")
business_comparison_df.to_csv(business_comparison_path, index=False)

print("Comparaison clean vs business :", business_comparison_path)
business_comparison_df


Comparaison clean vs business : artifacts\regression_clean_vs_business_comparison.csv


,dataset_type,model,mae_days,rmse_days,r2_score,business_features_used
0,clean,clean_linear_regression,6.922374,11.166817,0.984107,
1,business,business_linear_regression,6.940882,11.153257,0.984146,"machine_type, factory_id, production_line_id, ..."
2,clean,clean_gradient_boosting,6.980395,11.153323,0.984146,


## 6. Predictions finales et fichiers generes

On garde le modele avec la plus faible MAE, puis on calcule les predictions de maintenance.

In [9]:
best_model_name = results_df.iloc[0]["model"]
best_model = fitted_models[best_model_name]

df_flags["predicted_rul_days"] = np.clip(best_model.predict(X), 1, 365).round(1)
df_flags["predicted_degradation_rate_pct"] = ((1 - df_flags["predicted_rul_days"] / 365) * 100).clip(0, 100).round(1)
df_flags["recommended_maintenance_date"] = df_flags["timestamp"] + pd.to_timedelta(
    np.ceil(df_flags["predicted_rul_days"]), unit="D"
)
df_flags["maintenance_priority"] = pd.cut(
    df_flags["predicted_rul_days"],
    bins=[0, 7, 30, 90, 365],
    labels=["urgent", "high", "medium", "low"],
)

dataset_path = os.path.join(ARTIFACT_DIR, "regression_dataset_with_flags.csv")
comparison_path = os.path.join(ARTIFACT_DIR, "regression_with_flags_comparison.csv")
predictions_path = os.path.join(ARTIFACT_DIR, "maintenance_predictions_with_flags.csv")
model_path = os.path.join(ARTIFACT_DIR, "best_rul_regressor_with_flags.pkl")

df_flags.to_csv(dataset_path, index=False)
results_df.to_csv(comparison_path, index=False)
joblib.dump(best_model, model_path)

prediction_columns = [
    "timestamp",
    "machine_id",
    "predicted_rul_days",
    "predicted_degradation_rate_pct",
    "recommended_maintenance_date",
    "maintenance_priority",
    "degradation_rate_pct",
    "engineered_rul_days",
] + flag_columns

predictions = df_flags[prediction_columns].sort_values("predicted_rul_days")
predictions.to_csv(predictions_path, index=False)

print("Meilleur modele :", best_model_name)
print("Dataset enrichi :", dataset_path)
print("Comparaison modeles :", comparison_path)
print("Predictions :", predictions_path)
print("Modele :", model_path)

predictions.head(15)

Meilleur modele : LinearRegression
Dataset enrichi : artifacts\regression_dataset_with_flags.csv
Comparaison modeles : artifacts\regression_with_flags_comparison.csv
Predictions : artifacts\maintenance_predictions_with_flags.csv
Modele : artifacts\best_rul_regressor_with_flags.pkl


,timestamp,machine_id,predicted_rul_days,predicted_degradation_rate_pct,recommended_maintenance_date,maintenance_priority,degradation_rate_pct,engineered_rul_days,flag_temperature_high,flag_vibration_high,flag_sound_high,flag_oil_low,flag_coolant_low,flag_pressure_low,flag_coolant_flow_low,flag_heat_high,flag_power_high,flag_error_codes_high,flag_anomaly_high,flag_operational_hours_high
99971,2026-03-29 00:40:00,M04,1.0,99.7,2026-03-30 00:40:00,urgent,100.0,6.1,1,1,1,1,1,1,1,1,1,1,1,1
99943,2026-03-29 00:05:00,M04,1.0,99.7,2026-03-30 00:05:00,urgent,100.0,6.2,1,1,1,1,1,1,1,1,1,1,1,1
91607,2026-03-21 18:25:00,M04,1.0,99.7,2026-03-22 18:25:00,urgent,100.0,14.6,1,1,1,1,1,1,1,1,1,1,1,1
91567,2026-03-21 17:35:00,M04,1.0,99.7,2026-03-22 17:35:00,urgent,100.0,1.0,1,1,1,1,1,1,1,1,1,1,1,1
85703,2026-03-16 15:25:00,M04,1.0,99.7,2026-03-17 15:25:00,urgent,100.0,8.8,1,1,1,1,1,1,1,1,1,1,1,1
91539,2026-03-21 17:00:00,M04,1.0,99.7,2026-03-22 17:00:00,urgent,100.0,13.2,1,1,1,1,1,1,1,1,1,1,1,1
90147,2026-03-20 12:00:00,M04,1.0,99.7,2026-03-21 12:00:00,urgent,100.0,1.0,1,1,1,1,1,1,1,1,1,1,1,1
90151,2026-03-20 12:05:00,M04,1.0,99.7,2026-03-21 12:05:00,urgent,100.0,4.6,1,1,1,1,1,1,1,1,1,1,1,1
82007,2026-03-13 10:25:00,M04,1.0,99.7,2026-03-14 10:25:00,urgent,100.0,12.8,1,1,1,1,1,1,1,1,1,1,1,1
81919,2026-03-13 08:35:00,M04,1.0,99.7,2026-03-14 08:35:00,urgent,100.0,1.0,1,1,1,1,1,1,1,1,1,1,1,1


## Interpretation attendue

Avec les flags, le modele doit mieux apprendre car la cible RUL est construite a partir d'une logique industrielle explicite.

A presenter ainsi :

> Nous avons enrichi les donnees capteurs avec des flags de degradation, puis construit une cible RUL coherente avec ces signaux. Le modele de regression estime le nombre de jours restants avant maintenance. Ce RUL permet ensuite de calculer un taux de degradation et une date recommandee d'intervention.

Attention : cette cible est experimentale/simulee. Dans un cas industriel reel, elle doit etre remplacee par des historiques de pannes et de maintenances reels.

La section `clean vs business` permet aussi de justifier le choix final : si les variables metier ameliorent la MAE/R2, elles peuvent etre conservees comme features de contexte. Sinon, on garde les capteurs comme socle principal et les donnees business pour le reporting et l'interpretation. Le resultat est sauvegarde dans `artifacts/regression_clean_vs_business_comparison.csv`.
